In [ ]:
# -----------------------------导入python库------------------------------------
import os
import netCDF4 as nc
import pandas as pd
import numpy as np
import numpy.ma as ma
import xarray as xr
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.style as mplstyle
import cartopy.crs as ccrs
import netCDF4 as nc
from netCDF4 import Dataset
import shapefile
import time
from datetime import datetime, timedelta

import cmaps
BASE_PATH = "/home/yfdong/data/work/SMP"
DATA_PATH = "/raid61/yfdong/data/work/SMP"
import sys
# 将目标模块所在的路径添加到 sys.path 中
sys.path.append(os.path.join(BASE_PATH, "code", "Library"))
from MeteoVarPlot import draw_2DVAR, draw_Bias, draw_VARtemporal, draw_Bias_subplot,draw_2DVAR_subplot ,draw_2DuvVAR ,draw_uvBias_subplot ,draw_2DuvVAR_subplot
from MeteoChartPlot import add_TickGrid
from PreprocessVar import (
    get_MONTH_abbr, get_pentad, get_letter,
    preprocess_var, read_var,Events_configs,
    Read_VarsFromVarsDict, experiments_types,return_Event_Date,
    load_geodata, SubsetDomainConfigure ,create_domain_mask,get_letter,
)
Era5Land_VarResults={}
Wrfout_VarResults={}
Obs_DataPath = os.path.join(DATA_PATH, "obs")
# Era5Land_VarList = ['sp','u10','v10','t2m','q2m','tp','e','swvl1']
# Wrfout_VarList = ['PSFC', 'U10', 'V10', 'T2', 'Q2', 'RAINNC', 'ET', 'SH2O_Layer1']
# -----------------------------读取数据------------------------------------
LSM_VarList = ['SM'] #'Wind10m',
# LSM_VarList = ['SH']
# 读取地形数据
LAT, LON, LandMask, DEM = load_geodata(GeoPath =os.path.join(DATA_PATH, "Domain/geo_em.d01.nc") ,IF_LandMask=True)

Timedelta = 24  # 选取的wrfout时间间隔（hour）
CalibrationStatus = 'def'
LAYER = 0
START_YEAR = 2010
END_YEAR = 2017
YearNum = END_YEAR - START_YEAR + 1
StartReadMonth, EndReadMonth = 5, 8
StartDay, EndDay = 6, 31
DomainRegion = [107.7, 122.6, 21, 41.5]  # [west, east, south, north]


In [ ]:


VarUnit = {
    'SM': 'm³/m³',
    'ET': 'mm/day',
    'SH': 'W/m²',
    'Q2m': 'g/kg',
    'T2m': 'K',
    'PSFC': 'hPa',
    'Wind10m': 'm/s',
    'Prec': 'mm/day',
}
SOIL_DEPTH = '10cm'
# 修改部分1：在开始绘图前设置全局rcParam参数
plt.rcParams['axes.edgecolor'] = 'black'  # 边框颜色
plt.rcParams['axes.linewidth'] = 0.35     # 边框线宽
import xarray as xr
CtrlName, ExpName = 'Ctrl', 'FSM'

In [ ]:
# ------------------------------------------- 计算日历日的气候态平均 ---------------------------------
# 合并Obs和WRF

merge_df = pd.read_parquet("SM_merge_df.parquet")
merge_df.rename(columns={'SMCI*': 'SMCI'}, inplace=True)
NumericColumns = merge_df.select_dtypes(include=['number']).columns.tolist()
merge_df['MMDD'] = merge_df['Date'].dt.strftime('%m-%d')
ClimateTemporal_merge_df = merge_df.groupby(['ID', 'MMDD'])[NumericColumns].mean().reset_index()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
def compute_metric(obs, sim, metric_name):
    # 质控（也可在外部做）
    obs = np.asarray(obs, dtype=np.float64)
    sim = np.asarray(sim, dtype=np.float64)
    valid = ~(np.isnan(obs) | np.isnan(sim))
    if not np.any(valid):
        return np.nan
    obs, sim = obs[valid], sim[valid]
    if metric_name == 'RMSE':
        return np.sqrt(np.nanmean((sim - obs) ** 2))
    elif metric_name == 'BIAS':
        return np.nanmean(sim - obs)
    elif metric_name == 'CORR':
        if np.std(obs) == 0 or np.std(sim) == 0:
            return np.nan
        return np.corrcoef(obs, sim)[0, 1]
    elif metric_name == 'RMSE':
        r = np.corrcoef(obs, sim)[0, 1] if (np.std(obs) != 0 and np.std(sim) != 0) else np.nan
        if np.isnan(r) or np.mean(obs) == 0 or np.std(obs) == 0:
            return np.nan
        alpha = np.std(sim) / np.std(obs)
        beta = np.mean(sim) / np.mean(obs)
        return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)
    else:
        raise ValueError(f"Unknown metric: {metric_name}")
    
def PlotVar(Plot_result_dict , Yticks, subTitle,  ax1):
    # --------- 绘制土壤湿度模拟差异柱状图（左侧坐标轴）----------------------
    ax1.set_ylim(YlimMin, YlimMax)
    # ax1.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax1.set_yticks(Yticks)
    # ax1.set_ylabel(f"({VarUnit[VarName]})", fontsize=fontsize, labelpad =10)
    LineWidth = 2


    # 绘制其他土壤湿度数据
    ax1.plot(DateList, Plot_result_dict['ERA5-Land'], 
            color= "#CDBE70", 
            linewidth = LineWidth*0.9 , alpha=0.8, label='ERA5-Land', zorder=10, linestyle='--') #,zorder=10  
    ax1.plot(DateList, Plot_result_dict['GLDAS-Noah'], 
            color= "#698B22", 
            linewidth = LineWidth*0.9 , alpha=0.8, label='GLDAS-Noah', zorder=10, linestyle='--') #,zorder=10      
    ax1.plot(DateList, Plot_result_dict['CLDAS'], 
            color= "#CD5555", 
            linewidth = LineWidth*0.9 , alpha=0.8, label='CLDAS', zorder=10, linestyle='--') #,zorder=10     
    # ----------- 绘制观测（右侧坐标轴）----------------------
    ax1.plot(DateList, Plot_result_dict['OBS'], 
            color= "#363636", marker='o',markersize= 3, markeredgewidth=0.1,
            linewidth = LineWidth , alpha=0.8, label="OBS", zorder=10) #,zorder=10  
    ax1.plot(DateList, Plot_result_dict[CtrlName], 
            color= "#73A1C2", 
            linewidth = LineWidth , alpha=1, label='CTRL', zorder=10) #,zorder=10  
    ax1.plot(DateList, Plot_result_dict[ExpName], 
            color= "#FF8C69", 
            linewidth = LineWidth , alpha=1, label=ExpName, zorder=10) #,zorder=10  
    # 时间标签设置
    ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
    # 设置X轴范围
    ax1.set_xlim(DateList[0], DateList[-2])
    # 设置刻度标签字体大小
    ax1.tick_params(axis='x', labelsize=fontsize-1)  # 设置x轴刻度标签大小
    ax1.tick_params(axis='y', labelsize=fontsize-1)  # 设置y轴刻度标签大小
    # 调整刻度线参数：
    ax1.minorticks_on()
    # 显示副刻度线
    ax1.tick_params(axis="both", which="major", direction="in", width=0.9, length=5, color='#363636')
    # grid 设置网格线性
    ax1.grid(True, which="major", linestyle="--", color="gray", linewidth=0.75, alpha = 0.3)

    ax1.set_title(subTitle, fontsize= fontsize, x= 0., y = 1.02, 
                  loc='left', weight='normal',  alpha=1)
    pass
SubsetRegions = {
    'HHP':[35, 40, 114., 119],
    'LPT': [35, 40, 108, 114.],
    'QDW': [28, 35, 108, 112.5], # Qinling-Daba-Wushan
    'YHP': [28, 35, 112.5, 121], 
    'SEH': [23, 28, 110, 120] # Southeastern Hills
}
Xlim_dict = {
    'HHP': [0.12, 0.38],
    'LPT': [0.12, 0.38],
    'QDW': [0.12, 0.38],
    'YHP': [0.2, 0.45],
    'SEH': [0.2, 0.45]
}
Yticks_dict = {
    'HHP': [0.15, 0.25, 0.35],
    'LPT': [0.15, 0.25, 0.35],
    'QDW': [0.15, 0.25, 0.35],
    'YHP': [0.2, 0.3, 0.4],
    'SEH': [0.2, 0.3, 0.4]
}
Rows = 2
Cols = 3
fontsize = 13
fig, axes = plt.subplots(Rows, Cols, figsize=(Cols*2.6*1, Rows*2.6*1), dpi=500, 
                        constrained_layout=True, sharex=False)
VarName = 'SM'
Year = 2000

# 初始化存储 KGE 的列表
rmse_ctrl, rmse_exp, rmse_era5, rmse_gldas, rmse_cldas = [], [], [], [], []
Read_StartDate, Read_EndDate = datetime(Year, StartReadMonth, StartDay), datetime(Year, EndReadMonth, EndDay)
DateList = pd.date_range(start=Read_StartDate, end=Read_EndDate, freq='24h')
axes = axes.flatten()
for IDRegion, RegionName in enumerate(SubsetRegions):
    YlimMin, YlimMax = Xlim_dict[RegionName]
    Yticks = Yticks_dict[RegionName]
    Region = SubsetRegions[RegionName]
    RegionClimateTemporal_merge_df = ClimateTemporal_merge_df[(ClimateTemporal_merge_df['Latitude'] >= Region[0]) & (ClimateTemporal_merge_df['Latitude'] <= Region[1]) & (ClimateTemporal_merge_df['Longitude'] >= Region[2]) & (ClimateTemporal_merge_df['Longitude'] <= Region[3])]
    RegionClimateTemporal_merge_df_mean = RegionClimateTemporal_merge_df.groupby(['MMDD'])[NumericColumns].mean().reset_index()
    axeIndex = IDRegion
    subTitle = f"({get_letter(axeIndex)}) {RegionName}"
    PlotVar(RegionClimateTemporal_merge_df_mean, Yticks, subTitle, axes[axeIndex])
    IDx, IDy = np.unravel_index(axeIndex, (Rows, Cols))
    if IDy != 0:
        axes[axeIndex].set_yticklabels([])  # 清除 y 轴刻度标签
        axes[axeIndex].set_ylabel('')       # 清除 y 轴标签
    # 收集 KGE 值（在 PlotVar 内部已经计算，这里再次调用 compute_metric）
    ctrl_rmse = compute_metric(RegionClimateTemporal_merge_df_mean['OBS'], 
                              RegionClimateTemporal_merge_df_mean[CtrlName], 'RMSE')
    exp_rmse = compute_metric(RegionClimateTemporal_merge_df_mean['OBS'], 
                             RegionClimateTemporal_merge_df_mean[ExpName], 'RMSE')
    era5_rmse = compute_metric(RegionClimateTemporal_merge_df_mean['OBS'], 
                              RegionClimateTemporal_merge_df_mean['ERA5-Land'], 'RMSE')
    gldas_rmse = compute_metric(RegionClimateTemporal_merge_df_mean['OBS'], 
                               RegionClimateTemporal_merge_df_mean['GLDAS-Noah'], 'RMSE')
    cldas_rmse = compute_metric(RegionClimateTemporal_merge_df_mean['OBS'], 
                               RegionClimateTemporal_merge_df_mean['CLDAS'], 'RMSE')
    rmse_ctrl.append(ctrl_rmse)
    rmse_exp.append(exp_rmse)
    rmse_era5.append(era5_rmse)
    rmse_gldas.append(gldas_rmse)
    rmse_cldas.append(cldas_rmse)
    
    IDx, IDy = np.unravel_index(axeIndex, (Rows, Cols))
    if IDx == 0:
        axes[axeIndex].set_xticklabels([])
        axes[axeIndex].set_xlabel('')
# ---------- 绘制第六个子图：KGE 柱状图 ----------
ax_bar = axes[5]   # 最后一个子图
regions = list(SubsetRegions.keys())
x = np.arange(len(regions))  # x 位置
width = 0.15                # 柱宽
# 保证柱状图颜色与折线一致
colors = {
    'Ctrl': "#73A1C2",
    'Exp': "#FF8C69",
    'ERA5-Land': "#CDBE70",
    'GLDAS-Noah': "#698B22",
    'CLDAS': "#CD5555"
}

# 按指定顺序绘制：从左到右 ERA5, GLDAS, CLDAS, Ctrl, Exp
ax_bar.bar(x - 2*width, rmse_era5, width, color=colors['ERA5-Land'], label='ERA5-Land', zorder=3, alpha = 0.8)
ax_bar.bar(x - width,   rmse_gldas, width, color=colors['GLDAS-Noah'], label='GLDAS-Noah', zorder=3, alpha = 0.8)
ax_bar.bar(x,           rmse_cldas, width, color=colors['CLDAS'], label='CLDAS', zorder=3, alpha = 0.8)
ax_bar.bar(x + width,   rmse_ctrl, width, color=colors['Ctrl'], label=CtrlName, zorder=3, alpha = 0.8)
ax_bar.bar(x + 2*width, rmse_exp, width, color=colors['Exp'], label=ExpName, zorder=3)

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(regions, fontsize=fontsize-1)
# ax_bar.set_ylabel('RMSE', fontsize=fontsize)
ax_bar.set_title('(f) RMSE across regions', fontsize=fontsize, loc='left')
ax_bar.grid(axis='y', linestyle='--', alpha=0.3)
ax_bar.tick_params(labelsize=fontsize-1)
# 将 Y 轴刻度移至右侧
ax_bar.yaxis.tick_right()          # 刻度线移到右边
ax_bar.yaxis.set_label_position("right")   # Y 轴标签也移到右边
ax_bar.set_yticks([0, 0.05, 0.1])
# 🔴 新增：添加全局统一的 Y 轴标签
fig.supylabel(f"({VarUnit[VarName]})", fontsize=fontsize) 
# 创建统一的图例
lines, labels = fig.axes[-2].get_legend_handles_labels()
fig.legend( lines, labels,   
           fontsize= fontsize-1,        
            bbox_to_anchor=(0.88, 0.001),ncol=6, 
            columnspacing=0.5, 
            handlelength=1.0,
            framealpha=1, edgecolor='none',
)
# 设置子图间距
plt.subplots_adjust(wspace=0.1, hspace=0.1)  # 调整宽度和高度间距
plt.show()